In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv")

categorical = list(df.select_dtypes(include=['str', 'object']).columns)
numerical = list(df.select_dtypes(include=['float64', 'int64']).columns)
numerical.remove('converted')

def train(df_train, y_train, C=1.0):
    dict = df_train[categorical + numerical].to_dict(orient='records')
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dict)
    
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
    model.fit(X_train, y_train)
    
    return dv, model

def predict(df, dv, model):
    dict = df[numerical + categorical].to_dict(orient = 'records')
    
    X = dv.transform(dict)
    y_pred = model.predict_proba(X)[:, 1]
    return y_pred

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0)

print(categorical)
print(numerical)
print('Target: converted')

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

cats = ['lead_score', 'number_of_courses_viewed', 'interaction_count', 'annual_income']
for c in cats:
    score = roc_auc_score(y_train, df_train[c])
    if score < 0.5:
        score = roc_auc_score(y_train, -df_train[c])
    print(c, score)

dv, model = train(df_train, y_train, 1.0)
y_pred = predict(df_val, dv, model)

print(roc_auc_score(y_val, y_pred))

thresholds = np.linspace(0, 1, 101)
best_t = 0
f1_t = 0
min_abs = 1e9
max_f1 = -1
actual_positive = (y_val == 1)
actual_negative = (y_val == 0)
for t in thresholds:
    tp = ((y_pred >= t) & actual_positive).sum()
    fp = ((y_pred>=t) & actual_negative).sum()
    fn = ((y_pred<t) & actual_positive).sum()
    
    Precision = (tp)/(tp+fp) if (tp + fp) > 0 else 500
    Recall = (tp)/(tp+fn) if (tp + fn) > 0 else 250
    f1 = 2*((Precision*Recall)/(Precision+Recall))
    if f1>max_f1:
        max_f1 = f1
        f1_t = t
    if abs(Precision-Recall)<min_abs:
        min_abs = abs(Precision-Recall)
        best_t = t

print(best_t)
print(f1_t)


print("\n5")

kfold = KFold(n_splits=5, shuffle=True, random_state=1)
scores = []

for train_idx, val_idx in kfold.split(df_full_train):
    df_train_cv = df_full_train.iloc[train_idx]
    df_val_cv = df_full_train.iloc[val_idx]
    
    y_train_cv = df_train_cv.converted.values
    y_val_cv = df_val_cv.converted.values
    
    dv, model = train(df_train_cv, y_train_cv, C=1.0)
    y_pred_cv = predict(df_val_cv, dv, model)
    
    auc = roc_auc_score(y_val_cv, y_pred_cv)
    scores.append(auc)

print(f"(std): {np.std(scores).round(3)}")


print("\n6")
C_values = [0.000001, 0.001, 1.0]

for C in C_values:
    kfold = KFold(n_splits=5, shuffle=True, random_state=1)
    scores = []
    
    for train_idx, val_idx in kfold.split(df_full_train):
        df_train_cv = df_full_train.iloc[train_idx]
        df_val_cv = df_full_train.iloc[val_idx]
        
        y_train_cv = df_train_cv.converted.values
        y_val_cv = df_val_cv.converted.values
        
        dv, model = train(df_train_cv, y_train_cv, C=C)
        y_pred_cv = predict(df_val_cv, dv, model)
        
        auc = roc_auc_score(y_val_cv, y_pred_cv)
        scores.append(auc)
        
    mean_auc = np.mean(scores).round(3)
    std_auc = np.std(scores).round(3)
        
    print(f"C = {C:<10} | Mean AUC: {mean_auc} | Std: {std_auc}")
    
    

['lead_source', 'industry', 'employment_status', 'location']
['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
Target: converted
lead_score 0.7882254810906102
number_of_courses_viewed 0.7229894623989618
interaction_count 0.7649203047563227
annual_income 0.6081977776250831
0.7318222288173319
0.63
0.41000000000000003

5
(std): 0.007

6
C = 1e-06      | Mean AUC: 0.617 | Std: 0.019
C = 0.001      | Mean AUC: 0.749 | Std: 0.01
C = 1.0        | Mean AUC: 0.732 | Std: 0.007
